# 08. MAF で Foundry Agent Service を呼ぶエージェントを構築する

## コードの解説

`FoundryChatClient` は MAF と **Azure AI Foundry Agent Service** を統合するクライアントです。
`OpenAIChatCompletionClient` の代わりに `FoundryChatClient` を使うことで、
エージェントの実行が Foundry のバックエンドで管理されます。

### OpenAIChatCompletionClient との違い

| 項目 | OpenAIChatCompletionClient | FoundryChatClient |
|-----|--------------------------|------------------|
| バックエンド | Azure OpenAI のみ | Azure AI Foundry Agent Service |
| セッション管理 | アプリ側で手動管理 | Foundry が自動管理 |
| ツール統合 | カスタムツール / MCP | カスタム + Foundry 組み込みツール |
| 主な用途 | ローカル開発・シンプルなエージェント | 本番環境・エンタープライズ用途 |

### FoundryChatClient の仕組み

```
MAF Agent
  └─ FoundryChatClient
        │
        └─→ Azure AI Foundry Agent Service
                ├─ モデル推論（GPT-4o など）
                ├─ セッション・スレッド管理
                ├─ ツール実行管理
                └─ 監視・ロギング
```

### AgentSession によるマルチターン会話

`AgentSession` を使うことで、複数ターンの会話を通じてコンテキストを維持できます。
同じ `session` オブジェクトを `agent.run()` に渡すことで会話履歴が引き継がれます。

## Azure の操作手順

### 1. Azure AI Foundry プロジェクトの作成

1. [Azure AI Foundry ポータル](https://ai.azure.com) にアクセス
2. 「新しいプロジェクト」をクリックしてプロジェクトを作成
3. プロジェクトの「概要」ページから **エンドポイント URL** をコピー

### 2. モデルのデプロイ

1. プロジェクト内の「モデル + エンドポイント」→「モデルのデプロイ」をクリック
2. `gpt-4o` または `gpt-4o-mini` を選択してデプロイ
3. デプロイ名をメモ（例: `gpt-4o`）

### 3. .env ファイルへの設定追加

```env
AZURE_AI_PROJECT_ENDPOINT=https://<hub-name>.services.ai.azure.com/api/projects/<project-name>
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-4o
MCP_SERVER_URI=http://localhost:8000/mcp
```

### 4. MCP サーバーの起動（ツール使用時）

```bash
python mcp/mock_server.py
```

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity.aio import DefaultAzureCredential
from agent_framework import Agent, AgentSession, MCPStreamableHTTPTool
from agent_framework.foundry import FoundryChatClient

load_dotenv()

AZURE_AI_PROJECT_ENDPOINT      = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
AZURE_AI_MODEL_DEPLOYMENT_NAME = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
MCP_SERVER_URI                 = os.getenv("MCP_SERVER_URI", "http://localhost:8000/mcp")


# ---------------------------------------------------------------
# MCP ツールの準備
# FoundryChatClient でも MCPStreamableHTTPTool を使える
# ---------------------------------------------------------------
mcp_tool = MCPStreamableHTTPTool(
    name="game_shop_tools",
    url=MCP_SERVER_URI,
    description="Xbox ゲームショップの製品・注文・在庫情報を取得するツール群です。",
    timeout=120,
)


# ---------------------------------------------------------------
# FoundryChatClient を使った Foundry Agent Service エージェントの構築
#
# ポイント1: FoundryChatClient に credential と project_endpoint を渡す
# ポイント2: model には Foundry でデプロイしたモデル名を指定
# ポイント3: Agent の使い方は OpenAIChatCompletionClient と同じ
# ---------------------------------------------------------------
async def run_foundry_agent():
    async with DefaultAzureCredential() as credential:
        foundry_client = FoundryChatClient(
            credential=credential,
            project_endpoint=AZURE_AI_PROJECT_ENDPOINT,
            model=AZURE_AI_MODEL_DEPLOYMENT_NAME,
        )

        # Foundry Agent Service バックエンドのエージェントを作成
        async with Agent(
            client=foundry_client,
            name="FoundryGameShopAssistant",
            instructions=(
                "あなたは Foundry Agent Service で動作する Xbox ゲームショップのアシスタントです。"
                "MCP ツールを使って製品情報を検索し、お客様の質問に答えてください。"
            ),
            tools=[mcp_tool],
        ) as agent:

            # AgentSession でマルチターン会話（コンテキストが自動で引き継がれる）
            async with AgentSession() as session:
                print("=== Foundry Agent Service によるマルチターン会話 ===")

                # 1 ターン目
                r1 = await agent.run("Xbox のゲームを探しています。", session=session)
                print(f"エージェント: {r1}")
                print()

                # 2 ターン目（前のコンテキストを記憶）
                r2 = await agent.run("RPG ジャンルのものはありますか？", session=session)
                print(f"エージェント: {r2}")


await run_foundry_agent()